<a href="https://colab.research.google.com/github/aryan-chokshi/chatbot/blob/main/Model_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers accelerate sentence-transformers faiss-cpu pdfplumber pandas numpy pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.9 MB/s eta 0:00:00


In [2]:
import os, re, json, time
import numpy as np
import pandas as pd
import faiss
import pdfplumber
from sentence_transformers import SentenceTransformer

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
ZIP_PATH = "/content/drive/MyDrive/LLM (1).zip"

In [11]:
import zipfile, os

EXTRACT_DIR = "/content/LLM_extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

KNOWLEDGE_DIR = os.path.join(EXTRACT_DIR, "Knowledge")
SCREENSHOT_DIR = os.path.join(EXTRACT_DIR, "Screenshots")

print("Knowledge files:", os.listdir(KNOWLEDGE_DIR))
print("Screenshot files:", os.listdir(SCREENSHOT_DIR))

Knowledge files: ['GPWR Test_procedure_Power_decrease_100%_to_HSB_rev2C.pdf', 'Generic PWR Simulator - Training Guide  - November 2016.pdf', 'Generic PWR Training Presentation Updated 2016-2-10.pdf', 'GPWR Test_Procedure_Power_Increase_HSB_to_100%_rev2D.pdf']
Screenshot files: ['MTU1.png', 'MGP1.png', 'MGP3.png', 'IS.png', 'TCS1.png', 'MTU2.png', 'MTU4.png', 'Overview.png', 'MGP2.png', 'MTU5.png', 'MTU3.png']


In [12]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def truncate(text: str, max_chars: int = 2200) -> str:
    if text is None:
        return ""
    return text if len(text) <= max_chars else text[:max_chars]

def clean_text(t: str) -> str:
    t = re.sub(r"\s+", " ", t).strip()
    return t

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def pdf_to_chunks(pdf_path: str, chunk_chars: int = 1200, overlap: int = 200):
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            text = clean_text(text)
            if not text:
                continue

            # sliding window chunks
            start = 0
            while start < len(text):
                end = min(len(text), start + chunk_chars)
                chunk = text[start:end]
                chunks.append({
                    "pdf": os.path.basename(pdf_path),
                    "page": i + 1,
                    "text": chunk
                })
                start += (chunk_chars - overlap)
    return chunks

def build_corpus(knowledge_dir: str):
    all_chunks = []
    for fn in os.listdir(knowledge_dir):
        if fn.lower().endswith(".pdf"):
            path = os.path.join(knowledge_dir, fn)
            all_chunks.extend(pdf_to_chunks(path))
    return pd.DataFrame(all_chunks)

df_chunks = build_corpus(KNOWLEDGE_DIR)
df_chunks.head(), len(df_chunks)

(                                                 pdf  page  \
 0  GPWR Test_procedure_Power_decrease_100%_to_HSB...     1   
 1  GPWR Test_procedure_Power_decrease_100%_to_HSB...     2   
 2  GPWR Test_procedure_Power_decrease_100%_to_HSB...     2   
 3  GPWR Test_procedure_Power_decrease_100%_to_HSB...     3   
 4  GPWR Test_procedure_Power_decrease_100%_to_HSB...     4   
 
                                                 text  
 0  GENERIC PWR SIMULATOR TEST PROCEDURE Power Dec...  
 1  Power Decrease from Full Power Level to Hot St...  
 2  EY-SAE™, 3KEYSTUDENT™, 3KEYTOUCH™ and 3KEYRELA...  
 3  Power Decrease from Full Power Level to Hot St...  
 4  Power Decrease from Full Power Level to Hot St...  ,
 1032)

In [14]:
def build_faiss_index(texts):
    embs = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embs = embs.astype("float32")
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index, embs

index, _ = build_faiss_index(df_chunks["text"].tolist())
print("FAISS ready. Chunks:", len(df_chunks))

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

FAISS ready. Chunks: 1032


In [15]:
def retrieve(query: str, top_k: int = 3):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, ids = index.search(q_emb, top_k)
    rows = df_chunks.iloc[ids[0]].copy()
    rows["score"] = scores[0]
    return rows

def format_context(rows: pd.DataFrame, max_chars: int = 2400):
    parts = []
    for _, r in rows.iterrows():
        parts.append(f"[{r['pdf']} p.{r['page']}] {r['text']}")
    ctx = "\n\n".join(parts)
    return truncate(ctx, max_chars=max_chars)

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_text_model(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
        device_map="auto" if DEVICE=="cuda" else None
    )
    return tok, model

def generate_llm(tok, model, system_prompt: str, user_prompt: str,
                 max_new_tokens=256, temperature=0.0, top_p=0.9):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    chat = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(chat, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            top_p=top_p if temperature > 0 else 1.0,
            pad_token_id=tok.eos_token_id
        )
    text = tok.decode(out[0], skip_special_tokens=True)
    # return the tail (not perfect, but ok for comparison)
    return text.split(user_prompt)[-1].strip()

In [17]:
BASELINE_QUERIES = [
    "What are the main steps to increase power from HSB to 100%?",
    "What preconditions or checks must be satisfied before initiating a power increase?",
    "Which systems are involved in turbine startup during power increase, and what is their purpose?",
    "If the procedure does not specify an action, what should the operator do?"
]

SYSTEM_PROMPT_RAG = (
    "You are a nuclear procedure assistant for a Generic PWR simulator training environment. "
    "Answer ONLY using the provided CONTEXT. "
    "If the answer is not in the context, say: "
    "\"I don't know based on the provided procedures.\" "
    "Be concise and step-by-step."
)

def build_user_prompt(query: str, context: str):
    return f"CONTEXT:\n{context}\n\nQUESTION: {query}\n\nANSWER:"

In [18]:
TEXT_MODELS = [
    # baseline-ish small instruct
    "mistralai/Mistral-7B-Instruct-v0.3",
    # llama instruct (requires access approval on HF for Meta models)
    "meta-llama/Meta-Llama-3-8B-Instruct",
    # optional: another family
    # "Qwen/Qwen2.5-7B-Instruct"
]

In [22]:
!pip -q install huggingface_hub
from huggingface_hub import login
login()  # paste your HF token when prompted

In [23]:
results = []

for model_id in TEXT_MODELS:
    print("\n=== Loading:", model_id, "===")
    tok, model = load_text_model(model_id)

    for q in BASELINE_QUERIES:
        rows = retrieve(q, top_k=3)
        ctx = format_context(rows)

        prompt = build_user_prompt(q, ctx)

        t0 = time.time()
        ans = generate_llm(tok, model, SYSTEM_PROMPT_RAG, prompt, max_new_tokens=280, temperature=0.0)
        latency = time.time() - t0

        results.append({
            "model": model_id,
            "query": q,
            "latency_s": round(latency, 2),
            "top_context_sources": "; ".join([f"{r['pdf']} p.{r['page']}" for _, r in rows.iterrows()]),
            "answer": ans
        })

df_results = pd.DataFrame(results)
df_results.head()


=== Loading: mistralai/Mistral-7B-Instruct-v0.3 ===


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


=== Loading: meta-llama/Meta-Llama-3-8B-Instruct ===


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

,model,query,latency_s,top_context_sources,answer
0,mistralai/Mistral-7B-Instruct-v0.3,What are the main steps to increase power from...,6.01,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,1. Match flags on startup transformer 13.8 KV ...
1,mistralai/Mistral-7B-Instruct-v0.3,What preconditions or checks must be satisfied...,3.71,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,"Before initiating a power increase, the follow..."
2,mistralai/Mistral-7B-Instruct-v0.3,Which systems are involved in turbine startup ...,10.22,Generic PWR Training Presentation Updated 2016...,"During turbine startup during power increase, ..."
3,mistralai/Mistral-7B-Instruct-v0.3,"If the procedure does not specify an action, w...",3.54,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,"If the procedure does not specify an action, t..."
4,meta-llama/Meta-Llama-3-8B-Instruct,What are the main steps to increase power from...,6.28,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,"assistant\n\nBased on the provided procedure, ..."


In [24]:
def grounding_overlap(answer: str, context: str):
    a = set(re.findall(r"[A-Za-z]{4,}", answer.lower()))
    c = set(re.findall(r"[A-Za-z]{4,}", context.lower()))
    if not a:
        return 0.0
    return len(a & c) / len(a)

# compute overlap for each row (rebuild contexts quickly from saved sources is hard)
# instead, compute from the retrieved context for each row during generation
# So here's a re-run that stores overlap too:

results = []
for model_id in TEXT_MODELS:
    tok, model = load_text_model(model_id)
    for q in BASELINE_QUERIES:
        rows = retrieve(q, top_k=3)
        ctx = format_context(rows)
        prompt = build_user_prompt(q, ctx)

        t0 = time.time()
        ans = generate_llm(tok, model, SYSTEM_PROMPT_RAG, prompt, max_new_tokens=280, temperature=0.0)
        latency = time.time() - t0

        results.append({
            "model": model_id,
            "query": q,
            "latency_s": round(latency, 2),
            "grounding_overlap": round(grounding_overlap(ans, ctx), 3),
            "sources": "; ".join([f"{r['pdf']} p.{r['page']}" for _, r in rows.iterrows()]),
            "answer": ans
        })

df_results = pd.DataFrame(results)
df_results.sort_values(["model","query"]).head()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

,model,query,latency_s,grounding_overlap,sources,answer
7,meta-llama/Meta-Llama-3-8B-Instruct,"If the procedure does not specify an action, w...",0.44,0.400,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,assistant\n\nI don't know based on the provide...
4,meta-llama/Meta-Llama-3-8B-Instruct,What are the main steps to increase power from...,6.13,0.725,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,"assistant\n\nBased on the provided procedure, ..."
5,meta-llama/Meta-Llama-3-8B-Instruct,What preconditions or checks must be satisfied...,5.16,0.519,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,system\n\nYou are a nuclear procedure assistan...
6,meta-llama/Meta-Llama-3-8B-Instruct,Which systems are involved in turbine startup ...,8.24,0.709,Generic PWR Training Presentation Updated 2016...,"assistant\n\nBased on the provided context, du..."
3,mistralai/Mistral-7B-Instruct-v0.3,"If the procedure does not specify an action, w...",3.47,0.667,GPWR Test_Procedure_Power_Increase_HSB_to_100%...,"If the procedure does not specify an action, t..."


In [25]:
df_summary = (df_results
              .groupby("model")[["latency_s","grounding_overlap"]]
              .mean()
              .reset_index()
              .sort_values("grounding_overlap", ascending=False))
df_summary

,model,latency_s,grounding_overlap
1,mistralai/Mistral-7B-Instruct-v0.3,5.7650,0.72475
0,meta-llama/Meta-Llama-3-8B-Instruct,4.9925,0.58825


In [26]:
df_results.to_csv("model_comparison_results.csv", index=False)
df_summary.to_csv("model_comparison_summary.csv", index=False)

from google.colab import files
files.download("model_comparison_results.csv")
files.download("model_comparison_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>